In [ ]:
from pathlib import Path

from social_groups.directories import TRACK_FILE_NAME_COMPRESSED
from social_groups.general.tracking import TrackEntry, iter_jsonl_zst
from social_groups.reporting.parsing import (
    _ANSWER_OPTIONS,
    _ANSWER_PATTERNS,
    AnswerOptions,
    get_string_parser,
)
from social_groups.trialrunner.data_connectors.mmlu_pro_subset import (
    MMLUProSubsetConnector,
)


In [ ]:
TOOL_USAGE_DIR = Path(
    "/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/runs/macbook-link/baseline L with tool usage/2026-02-16-15-02-15"
)
NO_TOOL_USAGE_DIR = Path(
    "/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/multirun/final/heterogeneous_group_baseline/2026-01-28-11-18-38/heterogeneous_group_baseline_0"
)

parser = get_string_parser(
    _ANSWER_PATTERNS[AnswerOptions.letters_A_to_J],
    _ANSWER_OPTIONS[AnswerOptions.letters_A_to_J],
)

In [3]:
data = list(MMLUProSubsetConnector().iterate_data())

In [4]:
def by_question(a):
    return a.input.question


tool_iter = sorted(
    map(
        TrackEntry.model_validate,
        iter_jsonl_zst(TOOL_USAGE_DIR / TRACK_FILE_NAME_COMPRESSED),
    ),
    key=by_question,
)
no_tool_iter = sorted(
    map(
        TrackEntry.model_validate,
        iter_jsonl_zst(NO_TOOL_USAGE_DIR / TRACK_FILE_NAME_COMPRESSED),
    ),
    key=by_question,
)

data = sorted(data, key=lambda x: x.question)

for tool, no_tool, d in zip(tool_iter, no_tool_iter, data):
    assert tool.input.question == no_tool.input.question, (
                                                              tool.input.question,
                                                              no_tool.input.question,
                                                          ) == d.question

    print(tool.output.final_answer, parser(no_tool.output.final_answer), d.answer)

C G D
A A B
(A): Late Sixth Century BCE D A
C C C
D D F
A A H
C F C
None A A
E E G
G H B
C E E
B G B
C C J
None D E
D C C
F A A
E F A
H A E
G A I
H H D
9.30 × 10^{-15} C·m I I
B C C
A A D
None C C
H B H
J G C
G G H
A A F
None C B
A I D
None A H
F C F
None ___not_parsable___ B
163 H H
C C F
None I J
F F F
C C B
F F H
C H C
C D D
C C C
C H C
H I I
37 J J
G ___not_parsable___ J
B B B
D D G
A A D
F A A
D B D
D E C
B B B
None B B
D B C
F F J
None ___not_parsable___ H
B D F
D D D
None ___not_parsable___ J
D D E
G ___not_parsable___ B
F ___not_parsable___ F
I I I
F B B
A A A
None ___not_parsable___ J
D B D
E E E
A ___not_parsable___ E
None ___not_parsable___ E
D ___not_parsable___ B
B D D
I I I
None C B
E E E
A A C
E C H
A D E
C B F
B B B
D B D
D D D
I H I
F F F
F F F
J J J
B B B
G G G
A A E
H F F
A E F
F F E
C C D
A, C, H, I A I
D D D
D A D
C B D
E D J
G D F


In [5]:
tool_correct = 0
no_tool_correct = 0
total = 0

differ = 0
for tool, no_tool, d in zip(tool_iter, no_tool_iter, data):
    if tool.output.final_answer == d.answer:
        tool_correct += 1
    if parser(no_tool.output.final_answer) == d.answer:
        no_tool_correct += 1

    if parser(no_tool.output.final_answer) != tool.output.final_answer:
        differ += 1

    total += 1

print(f"Tool Correct: {tool_correct}/{total} ({(tool_correct / total) * 100:.2f}%)")
print(
    f"Normal Correct: {no_tool_correct}/{total} ({(no_tool_correct / total) * 100:.2f}%)"
)
print(f"Total Different answers: {differ}")

Tool Correct: 31/100 (31.00%)
Normal Correct: 35/100 (35.00%)
Total Different answers: 61
